In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize SparkSession
spark = SparkSession.builder.appName('Indian Food Analysis').getOrCreate()

# Load the dataset (update the filepath to match your directory)
filepath = 'D:/ABD0008/DATA/indian_food.csv'
df = spark.read.csv(filepath, header=True, inferSchema=True)

# Optional: View the schema to confirm column names
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- diet: string (nullable = true)
 |-- prep_time: integer (nullable = true)
 |-- cook_time: integer (nullable = true)
 |-- flavor_profile: string (nullable = true)
 |-- course: string (nullable = true)
 |-- state: string (nullable = true)



In [3]:
# Assuming the dish name is in a column called 'name'
unique_dishes_count = df.select('name').distinct().count()
print(f"Number of unique dishes: {unique_dishes_count}")

Number of unique dishes: 255


In [4]:
# Groups by state, counts them, and sorts in descending order to find the top state
df.filter(F.col('state') != '-1') \
  .groupBy('state').count() \
  .orderBy(F.desc('count')) \
  .limit(1) \
  .show()


+-------+-----+
|  state|count|
+-------+-----+
|Gujarat|   35|
+-------+-----+



In [5]:
karnataka_dishes = df.filter(F.col('state') == 'Karnataka').count()
print(f"Dishes from Karnataka: {karnataka_dishes}")

Dishes from Karnataka: 6


In [7]:
# Listing unique values for each column separately
print("Unique Flavor Profiles:")
df.select('flavor_profile').distinct().show()

print("Unique Courses:")
df.select('course').distinct().show()

# Alternatively, to list unique combinations of both:
# df.select('flavor_profile', 'course').distinct().show()

Unique Flavor Profiles:
+--------------+
|flavor_profile|
+--------------+
|            -1|
|         spicy|
|         sweet|
|          sour|
|        bitter|
+--------------+

Unique Courses:
+-----------+
|     course|
+-----------+
|    starter|
|    dessert|
|      snack|
|main course|
+-----------+



In [8]:
df.filter((F.col('course') == 'main course') & (F.col('state') != '-1')) \
  .groupBy('state').count() \
  .orderBy(F.desc('count')) \
  .limit(1) \
  .show()

+------+-----+
| state|count|
+------+-----+
|Punjab|   28|
+------+-----+



In [9]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define the state-to-region mapping
df_mapped = df.withColumn(
    "region",
    F.when(F.col("state").isin("Punjab", "Uttar Pradesh", "Uttarakhand", "NCT of Delhi", "Jammu & Kashmir", "Haryana"), "North")
    .when(F.col("state").isin("Rajasthan", "Maharashtra", "Gujarat", "Goa"), "West")
    .when(F.col("state").isin("Andhra Pradesh", "Karnataka", "Telangana", "Kerala", "Tamil Nadu"), "South")
    .when(F.col("state").isin("West Bengal", "Odisha", "Bihar"), "East")
    .when(F.col("state").isin("Assam", "Tripura", "Manipur", "Nagaland"), "North East")
    .when(F.col("state").isin("Madhya Pradesh", "Chhattisgarh"), "Central")
    .otherwise("-1")  # Catch missing (-1) or unrecognized states
)

# Verify it worked
df_mapped.select('name', 'state', 'region').show(5)

+--------------+-----------+------+
|          name|      state|region|
+--------------+-----------+------+
|    Balu shahi|West Bengal|  East|
|        Boondi|  Rajasthan|  West|
|Gajar ka halwa|     Punjab| North|
|        Ghevar|  Rajasthan|  West|
|   Gulab jamun|West Bengal|  East|
+--------------+-----------+------+
only showing top 5 rows



In [10]:
# Filtering out the '-1' missing values to get the true region count
unique_regions = df_mapped.filter(F.col('region') != '-1').select('region').distinct()

print(f"Number of unique regions: {unique_regions.count()}")
unique_regions.show()

Number of unique regions: 6
+----------+
|    region|
+----------+
|     South|
|   Central|
|      East|
|      West|
|North East|
|     North|
+----------+



In [11]:
df_mapped.filter(F.col('region') != '-1') \
    .groupBy('region').count() \
    .orderBy(F.desc('count')) \
    .show()

+----------+-----+
|    region|count|
+----------+-----+
|      West|   74|
|     South|   49|
|     North|   46|
|      East|   34|
|North East|   25|
|   Central|    3|
+----------+-----+



In [12]:
# Calculate the total number of dishes that have a valid region
total_mapped_dishes = df_mapped.filter(F.col('region') != '-1').count()

# Calculate percentage: (count / total_mapped_dishes) * 100
df_mapped.filter(F.col('region') != '-1') \
    .groupBy('region').count() \
    .withColumn('percentage', F.round((F.col('count') / total_mapped_dishes) * 100, 2)) \
    .orderBy(F.desc('percentage')) \
    .show()

+----------+-----+----------+
|    region|count|percentage|
+----------+-----+----------+
|      West|   74|     32.03|
|     South|   49|     21.21|
|     North|   46|     19.91|
|      East|   34|     14.72|
|North East|   25|     10.82|
|   Central|    3|       1.3|
+----------+-----+----------+



In [13]:
# Partition the data by region, then rank states by their dish count in descending order
windowRegion = Window.partitionBy('region').orderBy(F.desc('count'))

df_mapped.filter((F.col('region') != '-1') & (F.col('state') != '-1')) \
    .groupBy('region', 'state').count() \
    .withColumn('rank', F.rank().over(windowRegion)) \
    .filter(F.col('rank') == 1) \
    .drop('rank') \
    .orderBy('region') \
    .show()

+----------+--------------+-----+
|    region|         state|count|
+----------+--------------+-----+
|   Central|Madhya Pradesh|    2|
|      East|   West Bengal|   24|
|     North|        Punjab|   32|
|North East|         Assam|   21|
|     South|    Tamil Nadu|   20|
|      West|       Gujarat|   35|
+----------+--------------+-----+

